In [ ]:
import random
import pandas as pd
from datasets import load_dataset, Dataset
import time
import json

# ==============================================================================
# ✨ AI 코딩 튜토리얼: 나의 AI 캐릭터 프로파일러 만들기 ✨
# 👩‍🏫 튜터의 설명: 
# 이 데이터셋은 '한국의 다양한 페르소나(Persona)' 데이터를 담고 있어요. 
# 단순히 데이터를 훑어보는 것을 넘어, 각 캐릭터의 핵심 정보들을 모아서 
# 마치 AI가 분석한 것처럼 입체적인 '캐릭터 프로필'을 생성해 보는 것이 목표입니다. 
# 파이썬의 문자열 처리와 데이터 구조 이해를 한 번에 익힐 수 있는 최고의 실습입니다!
# ==============================================================================

# 데이터셋 이름 정의
DATASET_NAME = "nvidia/Nemotron-Personas-Korea"
SAMPLE_COUNT = 5 # 🔥 초보자 실습을 위해 5개의 샘플만 사용합니다!

# 🌟 데이터셋 메타 정보: 
# 이 데이터셋은 한글로 작성된 다양한 라이프스타일 페르소나(직업, 취미, 가족 등) 정보를 담고 있어, 
# 대규모 언어 모델(LLM)이 캐릭터를 생성하는 과정을 시뮬레이션하기에 최적입니다.
# 목표: 이 구조화된 데이터를 바탕으로, 매력적인 인물 설명을 생성하는 '프롬프트 엔지니어링'의 기초를 연습합니다.

# ------------------------------------------------------------------------------
# 1. 데이터 로딩 전략: 스트리밍(Streaming) 모드로 안전하게 로드하기
# ------------------------------------------------------------------------------
dataset = None
print("================================================================")
print(f"▶️ 데이터셋 로딩 시도: {DATASET_NAME}")

try:
    # 🎯 스트리밍 로드 시도 (가장 빠르고 효율적인 방법!)
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍(Streaming) 모드로 데이터셋을 로드했습니다. 메모리 효율 최고!")

except Exception as e:
    # 🚨 스트리밍 로드 실패 시 예외 처리: 일반 Dataset으로 다운로드 전환
    print(f"⚠️ 스트리밍 로드 중 오류 발생: {e}. 일반 Dataset 모드로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 성공! 일반 Dataset 모드로 데이터셋을 로드했습니다.")
    except Exception as e_fallback:
        print(f"❌ 치명적인 오류 발생: 데이터셋을 로드할 수 없습니다. ({e_fallback})")
        exit()

# ------------------------------------------------------------------------------
# 2. 샘플 데이터 준비 (Iterator 활용)
# ------------------------------------------------------------------------------
print("\n================================================================")
print(f"✨ {SAMPLE_COUNT}개의 샘플을 뽑아 분석을 시작합니다!")

# 💡 Constraint 준수: len() 사용 금지, .take() 패턴 사용
# 1. .take()를 사용하여 상위 K개 샘플을 가져옵니다. (Memory Efficient!)
sampled_dataset = dataset.take(SAMPLE_COUNT)

# 2. 반복자가 존재하면 (스트리밍 데이터셋일 가능성 높음)
if hasattr(sampled_dataset, "__iter__"):
    # Sample Iterator를 생성합니다.
    sample_iterator = iter(sampled_dataset)
else:
    # 일반 데이터셋인 경우, 리스트로 변환하여 반복자처럼 사용합니다.
    sample_iterator = iter(list(sampled_dataset))


# ------------------------------------------------------------------------------
# 3. 핵심 함수 정의: AI 캐릭터 프로파일러 (가장 창의적인 부분!)
# ------------------------------------------------------------------------------

def generate_character_profile(sample: dict) -> str:
    """
    샘플 딕셔너리(캐릭터 데이터)를 받아, 매력적인 캐릭터 프로필을 생성합니다.
    -> LLM의 역할을 사람이 직접 시뮬레이션해보는 과정입니다!
    """
    
    # 🔑 핵심 정보 추출 (필요한 Feature만 골라냅니다!)
    occupation = sample.get("occupation", "직업 정보 없음")
    age = sample.get("age", 0)
    hobbies = sample.get("hobbies_and_interests", "특별한 취미는 말씀해주시지 않았어요.")
    culinary = sample.get("culinary_persona", "요리 관련 특징 없음")
    sports = sample.get("sports_persona", "운동에 대한 열정은 감지되지 않았습니다.")
    
    # 📝 프로파일 템플릿 조합 (f-string 사용)
    profile = f"""
    ======================================
    ✨ AI 분석 프로필: 만렙 캐릭터 생성 ✨
    ======================================
    👤 이름: (분석 필요) - 이 데이터셋은 이름이 없어 분위기로 추측합니다.
    📅 나이: {age}세 ({sample.get('sex', '미상')} 기준)
    💼 핵심 직업: {occupation}
    
    💡 종합 분석 요약:
    이 분은 {occupation}이라는 전문성을 가진 개인이자, 
    {hobbies}와 같은 취미를 통해 예술적 감각을 충족하는 분으로 보입니다.
    
    🍽️ 미식가의 관점 (Culinary):
    ' {culinary} '라는 독특한 미식관을 가지고 계십니다. 맛과 스토리텔링을 중시하는 분일 것 같습니다.
    
    🏃 운동/활동적 측면 (Sports):
    스포츠 페르소나에서 {sports}와 같은 활력을 느낄 수 있습니다. 평소 활동량이 많을 것으로 예상됩니다.
    
    ✨ T.O.P (Traits Of Personality):
    직업적 전문성 + 취미적 예술성 + 건강한 활동성 = 입체적이고 다재다능한 캐릭터! 
    궁극의 AI 캐릭터가 완성되었습니다!
    """
    return profile.strip()

# ------------------------------------------------------------------------------
# 4. 실습 실행: 루프 돌며 프로필 생성하기
# ------------------------------------------------------------------------------

print("\n================================================================")
print("🚀 실습 시작: 샘플 캐릭터 프로파일 생성 (Iteration)")
print("================================================================")

# 💖 반복문을 통해 데이터를 하나씩 처리하고 결과를 출력합니다.
for i, sample in enumerate(sample_iterator):
    print(f"\n\n--- [샘플 {i+1} / {SAMPLE_COUNT}] 데이터를 분석합니다 ---")
    
    # ⚠️ 실수 방지: 데이터가 누락되거나 형식이 다를 수 있으므로, get() 메서드와 기본값을 사용합니다.
    try:
        # 함수 호출! ✨ 데이터 구조화 -> 자연어 생성
        profile = generate_character_profile(sample)
        print(profile)
    except Exception as e:
        print(f"⚠️ 샘플 {i+1} 처리 중 오류가 발생했습니다: {e}")

print("\n================================================================")
print("🎉 축하합니다! 캐릭터 프로파일러 실습을 성공적으로 마쳤습니다.")
print("👏👏👏 핵심은 데이터를 '잘 이해하고', 필요한 정보만 '골라내서', '매력적으로 조합'하는 능력입니다!")
print("이것이 바로 AI가 정보를 처리하는 원리의 기본 과정입니다!")